In [1]:
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 51.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 47.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 69.2 MB/s eta 0:00:00:00:0100:01


In [2]:
%%writefile whisper_engine.py
import os
import glob
import json
import multiprocessing as mp
import queue
import torch

AUDIO_DIR = "/kaggle/input/datasets/bumbleboo/aic26-b2-taylor/dataset/audio"
OUTPUT_DIR = "/kaggle/working/transcripts"

def whisper_worker(task_queue, gpu_id, output_dir):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    from faster_whisper import WhisperModel
    
    print(f"🚀 [GPU {gpu_id}] Đang tải model Whisper large-v3...")
    model = WhisperModel(
        "large-v3", 
        device="cuda", 
        compute_type="float16" 
    )
    print(f"✅ [GPU {gpu_id}] Model đã sẵn sàng!")

    while True:
        try:
            audio_path = task_queue.get(timeout=10)
        except queue.Empty:
            break

        if audio_path is None:
            print(f"🛑 [GPU {gpu_id}] Nhận tín hiệu dừng. Nghỉ ngơi thôi!")
            break

        video_id = os.path.basename(audio_path).replace(".wav", "")
        out_json_path = os.path.join(output_dir, f"{video_id}.json")
        
        if os.path.exists(out_json_path):
            print(f"⏩ [GPU {gpu_id}] Đã tồn tại {video_id}, bỏ qua...")
            continue

        print(f"🎙️ [GPU {gpu_id}] Đang Transcribe: {video_id}...")
        
        try:
            segments, info = model.transcribe(
                audio_path, 
                language="vi", 
                beam_size=5,
                vad_filter=True
            )
            
            transcript_data = []
            for segment in segments:
                text = segment.text.strip()
                if text:
                    transcript_data.append({
                        "start_pts": round(segment.start, 3),
                        "end_pts": round(segment.end, 3),
                        "text": text
                    })
            
            with open(out_json_path, "w", encoding="utf-8") as f:
                json.dump(transcript_data, f, ensure_ascii=False, indent=2)
                
            print(f"✅ [GPU {gpu_id}] Xong {video_id}!")
            
        except Exception as e:
            print(f"❌ [GPU {gpu_id}] Lỗi transcribe {video_id}: {e}")

def main():
    mp.set_start_method('spawn', force=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    audio_files = sorted(glob.glob(os.path.join(AUDIO_DIR, "*.wav")))
    
    num_gpus = torch.cuda.device_count()
    if num_gpus == 0:
        num_gpus = 1

    print(f"✅ Tìm thấy {len(audio_files)} file audio.")
    print(f"🚀 Phân bổ công việc vào Queue cho {num_gpus} GPU...")

    task_queue = mp.Queue()
    
    for audio_path in audio_files:
        task_queue.put(audio_path)

    for _ in range(num_gpus):
        task_queue.put(None)

    processes = []
    
    for gpu_id in range(num_gpus):
        p = mp.Process(
            target=whisper_worker, 
            args=(task_queue, gpu_id, OUTPUT_DIR)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print("\n🎉 HOÀN TẤT TRÍCH XUẤT AUDIO!")

if __name__ == '__main__':
    main()

Writing whisper_engine.py


In [3]:
!python whisper_engine.py

✅ Tìm thấy 2 file audio.
🚀 Phân bổ công việc vào Queue cho 2 GPU...
🚀 [GPU 1] Đang tải model Whisper large-v3...
🚀 [GPU 0] Đang tải model Whisper large-v3...
✅ [GPU 0] Model đã sẵn sàng!
🎙️ [GPU 0] Đang Transcribe: L21_V001...
✅ [GPU 1] Model đã sẵn sàng!
🎙️ [GPU 1] Đang Transcribe: L21_V002...
✅ [GPU 1] Xong L21_V002!
🛑 [GPU 1] Nhận tín hiệu dừng. Nghỉ ngơi thôi!
✅ [GPU 0] Xong L21_V001!
🛑 [GPU 0] Nhận tín hiệu dừng. Nghỉ ngơi thôi!

🎉 HOÀN TẤT TRÍCH XUẤT AUDIO!


In [4]:
!zip -r transcripts.zip /kaggle/working/transcripts

  adding: kaggle/working/transcripts/ (stored 0%)
  adding: kaggle/working/transcripts/L21_V002_transcript.json (deflated 75%)
  adding: kaggle/working/transcripts/L21_V001_transcript.json (deflated 75%)
